In [ ]:
import signalflow as sf
from pathlib import Path
from datetime import datetime

spot_store = sf.data.raw_store.DuckDbSpotStore(db_path=Path("test.duckdb"))

loader = sf.data.source.BinanceSpotLoader(store=spot_store)
await loader.download(
    pairs=["BTCUSDT", "ETHUSDT", "SOLUSDT", "BNBUSDT", "XRPUSDT"],
    start=datetime(2026, 1, 1),
    end=datetime(2026, 2, 23),
)

raw_data = sf.data.RawDataFactory.from_duckdb_spot_store(
    spot_store_path=Path("test.duckdb"),
    pairs=["BTCUSDT", "ETHUSDT", "SOLUSDT"],
    start=datetime(2026, 2, 1),
    end=datetime(2026, 2, 23),
    data_types=["spot"],
)
raw_data_view = sf.core.RawDataView(raw_data)

In [ ]:

from signalflow.ta import SmaSmooth

feature_type = "smooth/sma"
feture_params = {"period": 120}

feature = sf.get_component(sf.SfComponentType.FEATURE, feature_type)(**feture_params)
feature2 = SmaSmooth(period=360)
df = sf.FeaturePipeline(features=[feature, feature2]).run(raw_data_view)
df

In [ ]:
import plotly.graph_objects as go

btc = df.filter(df["pair"] == "BTCUSDT").sort("timestamp")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=btc["timestamp"].to_list(),
    y=btc["close"].to_list(),
    name="Close",
    line=dict(color="#888", width=1),
))
fig.add_trace(go.Scatter(
    x=btc["timestamp"].to_list(),
    y=btc["close_sma_120"].to_list(),
    name="SMA 120",
    line=dict(color="#2E86AB", width=1.5),
))
fig.add_trace(go.Scatter(
    x=btc["timestamp"].to_list(),
    y=btc["close_sma_360"].to_list(),
    name="SMA 360",
    line=dict(color="#E94F37", width=1.5),
))
fig.update_layout(
    title="BTCUSDT — Close / SMA 120 / SMA 360",
    xaxis_title="Time",
    yaxis_title="Price (USDT)",
    height=600,
    template="plotly_white",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
)
fig.show()